In [59]:
import pandas as pd
import numpy as np
import torch
import json
from pathlib import Path

In [60]:
# ------------------------------------------------------------
# SMILES-Zeichen-Wörterbuch wie bei AttentionDTA
# ------------------------------------------------------------

CHARISOSMISET = {
    "#": 29, "%": 30, ")": 31, "(": 1, "+": 32, "-": 33, "/": 34, ".": 2,
    "1": 35, "0": 3, "3": 36, "2": 4, "5": 37, "4": 5, "7": 38, "6": 6,
    "9": 39, "8": 7, "=": 40, "A": 41, "@": 8, "C": 42, "B": 9, "E": 43,
    "D": 10, "G": 44, "F": 11, "I": 45, "H": 12, "K": 46, "M": 47, "L": 13,
    "O": 48, "N": 14, "P": 15, "S": 49, "R": 16, "U": 50, "T": 17, "W": 51,
    "V": 18, "Y": 52, "[": 53, "Z": 19, "]": 54, "\\": 20, "a": 55, "c": 56,
    "b": 21, "e": 57, "d": 22, "g": 58, "f": 23, "i": 59, "h": 24, "m": 60,
    "l": 25, "o": 61, "n": 26, "s": 62, "r": 27, "u": 63, "t": 28, "y": 64
}


# ------------------------------------------------------------
# SMILES-Encoding bleibt unverändert
# ------------------------------------------------------------

def label_smiles(smiles, max_len=100):
    """
    Encodiert einen SMILES-String zeichenweise mit dem gleichen
    CHARISOSMISET-Wörterbuch wie im AttentionDTA-Baseline-Modell.
    """
    encoded = np.zeros(max_len, dtype=np.int64)

    for i, char in enumerate(str(smiles)[:max_len]):
        encoded[i] = CHARISOSMISET.get(char, 0)

    return encoded

In [61]:


#load data from CSV with unicode encoding
train_df = pd.read_csv("C:\\Users\\hempe\\Studium\\Masterthesis\\Repository\\Masterthesis\\data\\processed\\train_data.csv", sep=',')

In [62]:
train_df.head()

,Ligand SMILES,BindingDB Target Chain Sequence 1,IC50 (nM)
0,CC(C)Nc1cccnc1N1CCN(CC1)C(=O)c1cc2ccc(C=O)cc2[...,PISPIETVPVKLKPGMDGPKVKQWPLTEEKIKALVEICTEMEKEGK...,1400.0
1,[O-][N+](=O)c1ccc2N(Cc3ccccc3)C(=O)C(=O)c2c1,MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHL...,99500.0
2,CCOC(=O)c1c(Cc2cccc(Cl)c2)[nH]c2c1cc(O)c1ncccc21,MPSYTVTVATGSQWFAGTDDYIYLSLVGSAGCSEKHLLDKPFYNDF...,580.0
3,ON1C(=O)C(=C(C1=O)c1c[nH]c2ccccc12)c1c[nH]c2cc...,MADPAAGPPPSEGEESTVRFARKGALRQKNVHEVKNHKFTARFFKQ...,28900.0
4,c1c([nH]c2nccnc12)-c1cccc2ccccc12,MSGRPRTTSFAESCKPVQQPSAFGSMKVSRDKDGSKVTTVVATPGQ...,27000.0


In [63]:
sequence = train_df["BindingDB Target Chain Sequence 1"].iloc[0]

In [64]:
sequence_long = 'hi'

In [65]:
sequence_long = 'MPSYTVTVATGSQWFAGTDDYIYLSLVGSAGCSEKHLLDKPFYNDFMESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLMESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNRVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNNAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNNPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNRVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFMESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIG'

In [66]:
sequence

'PISPIETVPVKLKPGMDGPKVKQWPLTEEKIKALVEICTEMEKEGKISKIGPENPYNTPVFAIKKKDSTKWRKLVDFRELNKRTQDFWEVQLGIPHPAGLKKKKSVTVLDVGDAYFSVPLDEDFRKYTAFTIPSINNETPGIRYQYNVLPQGWKGSPAIFQSSMTKILEPFRKQNPDIVIYQYMDDLYVGSDLEIGQHRTKIEELRQHLLRWGLTTPDKKHQKEPPFLWMGYELHPDKWTVQPIVLPEKDSWTVNDIQKLVGKLNWASQIYPGIKVRQLCKLLRGTKALTEVIPLTEEAELELAENREILKEPVHGVYYDPSKDLIAEIQKQGQGQWTYQIYQEPFKNLKTGKYARMRGAHTNDVKQLTEAVQKITTESIVIWGKTPKFKLPIQKETWETWWTEYWQATWIPEWEFVNTPPLVKLWYQLEKEPIVGAETFYVDGAANRETKLGKAGYVTNRGRQKVVTLTDTTNQKTELQAIYLALQDSGLEVNIVTDSQYALGIIQAQPDQSESELVNQIIEQLIKKEKVYLAWVPAHKGIGGNEQVDKLVSAGIRKVL'

In [67]:
long_three_mers_list = []

for i in range(0, len(sequence_long)-2, 1):
    three_mer = sequence_long[i:i+3]
    long_three_mers_list.append(three_mer)  

In [68]:
long_three_mers_list

['MPS',
 'PSY',
 'SYT',
 'YTV',
 'TVT',
 'VTV',
 'TVA',
 'VAT',
 'ATG',
 'TGS',
 'GSQ',
 'SQW',
 'QWF',
 'WFA',
 'FAG',
 'AGT',
 'GTD',
 'TDD',
 'DDY',
 'DYI',
 'YIY',
 'IYL',
 'YLS',
 'LSL',
 'SLV',
 'LVG',
 'VGS',
 'GSA',
 'SAG',
 'AGC',
 'GCS',
 'CSE',
 'SEK',
 'EKH',
 'KHL',
 'HLL',
 'LLD',
 'LDK',
 'DKP',
 'KPF',
 'PFY',
 'FYN',
 'YND',
 'NDF',
 'DFM',
 'FME',
 'MES',
 'ESL',
 'SLV',
 'LVP',
 'VPG',
 'PGF',
 'GFN',
 'FNE',
 'NEK',
 'EKT',
 'KTH',
 'THV',
 'HVQ',
 'VQL',
 'QLS',
 'LSL',
 'SLP',
 'LPV',
 'PVL',
 'VLQ',
 'LQV',
 'QVR',
 'VRD',
 'RDV',
 'DVL',
 'VLV',
 'LVR',
 'VRG',
 'RGF',
 'GFG',
 'FGD',
 'GDS',
 'DSV',
 'SVE',
 'VEE',
 'EEV',
 'EVL',
 'VLS',
 'LSE',
 'SEA',
 'EAR',
 'ARQ',
 'RQH',
 'QHL',
 'HLM',
 'LME',
 'MES',
 'ESL',
 'SLV',
 'LVP',
 'VPG',
 'PGF',
 'GFN',
 'FNE',
 'NEK',
 'EKT',
 'KTH',
 'THV',
 'HVQ',
 'VQL',
 'QLS',
 'LSL',
 'SLP',
 'LPV',
 'PVL',
 'VLQ',
 'LQV',
 'QVR',
 'VRD',
 'RDV',
 'DVL',
 'VLV',
 'LVR',
 'VRG',
 'RGF',
 'GFG',
 'FGD',
 'GDS',
 'DSV',


In [69]:
three_mers_list = []

for i in range(0, len(sequence)-2, 1):
    three_mer = sequence[i:i+3]
    three_mers_list.append(three_mer)   

In [70]:
three_mers_list

['PIS',
 'ISP',
 'SPI',
 'PIE',
 'IET',
 'ETV',
 'TVP',
 'VPV',
 'PVK',
 'VKL',
 'KLK',
 'LKP',
 'KPG',
 'PGM',
 'GMD',
 'MDG',
 'DGP',
 'GPK',
 'PKV',
 'KVK',
 'VKQ',
 'KQW',
 'QWP',
 'WPL',
 'PLT',
 'LTE',
 'TEE',
 'EEK',
 'EKI',
 'KIK',
 'IKA',
 'KAL',
 'ALV',
 'LVE',
 'VEI',
 'EIC',
 'ICT',
 'CTE',
 'TEM',
 'EME',
 'MEK',
 'EKE',
 'KEG',
 'EGK',
 'GKI',
 'KIS',
 'ISK',
 'SKI',
 'KIG',
 'IGP',
 'GPE',
 'PEN',
 'ENP',
 'NPY',
 'PYN',
 'YNT',
 'NTP',
 'TPV',
 'PVF',
 'VFA',
 'FAI',
 'AIK',
 'IKK',
 'KKK',
 'KKD',
 'KDS',
 'DST',
 'STK',
 'TKW',
 'KWR',
 'WRK',
 'RKL',
 'KLV',
 'LVD',
 'VDF',
 'DFR',
 'FRE',
 'REL',
 'ELN',
 'LNK',
 'NKR',
 'KRT',
 'RTQ',
 'TQD',
 'QDF',
 'DFW',
 'FWE',
 'WEV',
 'EVQ',
 'VQL',
 'QLG',
 'LGI',
 'GIP',
 'IPH',
 'PHP',
 'HPA',
 'PAG',
 'AGL',
 'GLK',
 'LKK',
 'KKK',
 'KKK',
 'KKS',
 'KSV',
 'SVT',
 'VTV',
 'TVL',
 'VLD',
 'LDV',
 'DVG',
 'VGD',
 'GDA',
 'DAY',
 'AYF',
 'YFS',
 'FSV',
 'SVP',
 'VPL',
 'PLD',
 'LDE',
 'DED',
 'EDF',
 'DFR',
 'FRK',
 'RKY',


In [71]:
    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

In [72]:
for kmer in long_three_mers_list:
    if kmer not in vocab:
        vocab[kmer] = len(vocab)

In [73]:
len(vocab)

4049

In [74]:
vocab

{'<PAD>': 0,
 '<UNK>': 1,
 'MPS': 2,
 'PSY': 3,
 'SYT': 4,
 'YTV': 5,
 'TVT': 6,
 'VTV': 7,
 'TVA': 8,
 'VAT': 9,
 'ATG': 10,
 'TGS': 11,
 'GSQ': 12,
 'SQW': 13,
 'QWF': 14,
 'WFA': 15,
 'FAG': 16,
 'AGT': 17,
 'GTD': 18,
 'TDD': 19,
 'DDY': 20,
 'DYI': 21,
 'YIY': 22,
 'IYL': 23,
 'YLS': 24,
 'LSL': 25,
 'SLV': 26,
 'LVG': 27,
 'VGS': 28,
 'GSA': 29,
 'SAG': 30,
 'AGC': 31,
 'GCS': 32,
 'CSE': 33,
 'SEK': 34,
 'EKH': 35,
 'KHL': 36,
 'HLL': 37,
 'LLD': 38,
 'LDK': 39,
 'DKP': 40,
 'KPF': 41,
 'PFY': 42,
 'FYN': 43,
 'YND': 44,
 'NDF': 45,
 'DFM': 46,
 'FME': 47,
 'MES': 48,
 'ESL': 49,
 'LVP': 50,
 'VPG': 51,
 'PGF': 52,
 'GFN': 53,
 'FNE': 54,
 'NEK': 55,
 'EKT': 56,
 'KTH': 57,
 'THV': 58,
 'HVQ': 59,
 'VQL': 60,
 'QLS': 61,
 'SLP': 62,
 'LPV': 63,
 'PVL': 64,
 'VLQ': 65,
 'LQV': 66,
 'QVR': 67,
 'VRD': 68,
 'RDV': 69,
 'DVL': 70,
 'VLV': 71,
 'LVR': 72,
 'VRG': 73,
 'RGF': 74,
 'GFG': 75,
 'FGD': 76,
 'GDS': 77,
 'DSV': 78,
 'SVE': 79,
 'VEE': 80,
 'EEV': 81,
 'EVL': 82,
 'VLS': 83

In [75]:
three_mers_encoded_list = []

for kmer in three_mers_list:
    if kmer in vocab:
        three_mers_encoded_list.append(vocab[kmer])
    else:
        three_mers_encoded_list.append(vocab["<UNK>"])


In [76]:
for i in three_mers_encoded_list:
    if i != 1:
        print(i)


1864
3478
1295
516
2913
2687
638
639
2994
1663
2037
2734
2735
981
3197
2681
2168
747
748
646
1113
1114
705
3445
100
3832
2237
1571
1166
955
3986
386
1376
4029
2672
3596
3436
412
1671
3279
3480
2658
768
3383
3660
1702
328
1145
3823
1654
3475
1216
687
3879
2971
214
218
1001
2394
2357
3332
810
60
1313
858
3533
425
1144
328
328
2603
815
2490
7
1187
641
3977
493
3136
1440
2434
2180
2284
1715
2647
822
887
198
2212
1226
2867
3393
1833
2298
1196
2644
2690
1256
2909
3532
2170
2028
107
108
2805
2300
2289
4039
2390
3847
1355
1369
3484
2726
2212
1107
2887
2888
977
2120
1608
3505
3132
444
28
2198
606
2320
480
3151
1369
3673
3674
3675
2693
88
89
37
166
1413
2650
2908
1111
3074
484
2727
1725
288
3327
1131
1132
2548
952
1587
1634
107
3412
2839
2336
2568
1038
1673
3075
687
27
1765
496
3174
1179
3532
1230
1341
2385
3019
1746
1916
1629
166
3931
3352
1219
705
2923
747
745
1332
1158
2649
2168
747
748
978
3988
137
138
3801
3802
417
3511
481
1682
2373
4043
3630
268
1320
3043
3889
1395
2003
244
3886
1975
1092

In [77]:
three_mers_list[:10]

['PIS', 'ISP', 'SPI', 'PIE', 'IET', 'ETV', 'TVP', 'VPV', 'PVK', 'VKL']

In [78]:
def build_protein_kmer_vocab(protein_sequences, k=3):
    """
    Erstellt ein Vocabulary für Protein-k-mers aus den Trainingssequenzen.

    0 = Padding
    1 = Unknown Token für k-mers, die im Testset vorkommen,
        aber im Trainingsvokabular nicht enthalten sind.
    """
    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    for sequence in protein_sequences:
        kmers = protein_to_kmers(sequence, k=k)

        for kmer in kmers:
            if kmer not in vocab:
                vocab[kmer] = len(vocab)

    return vocab

In [ ]:
three_mers_list = []

for i in range(0, len(sequence), 3):
    three_mer = sequence[i:i+3]
    three_mers_list.append(three_mer)

In [ ]:
three_mers_list[:10]

['PIS', 'PIE', 'TVP', 'VKL', 'KPG', 'MDG', 'PKV', 'KQW', 'PLT', 'EEK']

In [ ]:

# ------------------------------------------------------------
# Protein-3-mer-Encoding nach WideDTA-Idee
# ------------------------------------------------------------

def protein_to_kmers(sequence, k=3):
    """
    Zerlegt eine Proteinsequenz in überlappende k-mer-Wörter.

    Beispiel:
    MTVKTE -> MTV, TVK, VKT, KTE
    """
    sequence = str(sequence)

    if len(sequence) < k:
        return []

    return [sequence[i:i + k] for i in range(len(sequence) - k + 1)]


def build_protein_kmer_vocab(protein_sequences, k=3):
    """
    Erstellt ein Vocabulary für Protein-k-mers aus den Trainingssequenzen.

    0 = Padding
    1 = Unknown Token für k-mers, die im Testset vorkommen,
        aber im Trainingsvokabular nicht enthalten sind.
    """
    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    for sequence in protein_sequences:
        kmers = protein_to_kmers(sequence, k=k)

        for kmer in kmers:
            if kmer not in vocab:
                vocab[kmer] = len(vocab)

    return vocab


def label_protein_kmers(sequence, vocab, max_len=1200, k=3):
    """
    Encodiert eine Proteinsequenz als Sequenz von k-mer-IDs.

    Die Ausgabe hat eine feste Länge von max_len:
    - längere Sequenzen werden abgeschnitten
    - kürzere Sequenzen werden mit 0 gepaddet
    """
    encoded = np.zeros(max_len, dtype=np.int64)

    kmers = protein_to_kmers(sequence, k=k)

    for i, kmer in enumerate(kmers[:max_len]):
        encoded[i] = vocab.get(kmer, vocab["<UNK>"])

    return encoded


# ------------------------------------------------------------
# Encoding-Funktion für Train und Test
# ------------------------------------------------------------

def encode_and_save(input_csv, output_path, protein_vocab, k=3):
    """
    Encodiert SMILES wie bisher zeichenweise und Proteinsequenzen
    als WideDTA-artige 3-mer-Wörter. Anschließend werden die Tensoren
    als .pt-Datei gespeichert.
    """

    # Daten laden
    df = pd.read_csv(input_csv, sep=",", encoding="utf-8")

    # SMILES-Encoding bleibt identisch zur AttentionDTA-Baseline
    df["smiles_encoded"] = df["Ligand SMILES"].apply(label_smiles)

    # Protein-Encoding als 3-mer-Wörter
    df["protein_encoded"] = df["BindingDB Target Chain Sequence 1"].apply(
        lambda seq: label_protein_kmers(
            sequence=seq,
            vocab=protein_vocab,
            max_len=1200,
            k=k
        )
    )

    # IC50 in pIC50 transformieren
    df["pIC50"] = -np.log10(df["IC50 (nM)"].astype(float) * 1e-9)

    # Arrays vorbereiten
    X_smiles = np.stack(df["smiles_encoded"].values)
    X_protein = np.stack(df["protein_encoded"].values)
    y = df["pIC50"].values.astype(np.float32)

    # PyTorch-Tensoren erstellen
    X_smiles_tensor = torch.tensor(X_smiles, dtype=torch.long)
    X_protein_tensor = torch.tensor(X_protein, dtype=torch.long)
    y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    # Tensoren speichern
    torch.save(
        {
            "X_smiles": X_smiles_tensor,
            "X_protein": X_protein_tensor,
            "y": y_tensor
        },
        output_path
    )

    print(f"Encoded data saved to: {output_path}")
    print("X_smiles:", X_smiles_tensor.shape)
    print("X_protein:", X_protein_tensor.shape)
    print("y:", y_tensor.shape)


# ------------------------------------------------------------
# Pfade
# ------------------------------------------------------------

train_csv = "data/processed/train_data.csv"
test_csv = "data/processed/test_data.csv"

encoded_train_path = "data/encoded/encoded_train_protein_3mer.pt"
encoded_test_path = "data/encoded/encoded_test_protein_3mer.pt"

vocab_path = "data/encoded/protein_3mer_vocab.json"

Path("data/encoded").mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Vocabulary nur auf Trainingsdaten erstellen
# ------------------------------------------------------------

train_df = pd.read_csv(train_csv, sep=",", encoding="utf-8")

protein_vocab = build_protein_kmer_vocab(
    protein_sequences=train_df["BindingDB Target Chain Sequence 1"],
    k=3
)

print("Protein 3-mer vocabulary size:", len(protein_vocab))

# Vocabulary speichern, damit später exakt dasselbe Mapping nachvollziehbar ist
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(protein_vocab, f, indent=2)

print(f"Protein vocabulary saved to: {vocab_path}")


# ------------------------------------------------------------
# Train- und Testdaten encodieren
# ------------------------------------------------------------

encode_and_save(
    input_csv=train_csv,
    output_path=encoded_train_path,
    protein_vocab=protein_vocab,
    k=3
)

encode_and_save(
    input_csv=test_csv,
    output_path=encoded_test_path,
    protein_vocab=protein_vocab,
    k=3
)